# RePix · DreamLite Mobile ONNX export

This notebook exports the **UNet and VAE only**. It does not export Qwen3-VL or produce an Android APK. The export runs here as a build step; the intended RePix inference remains on-device/offline.

**Before running:** DreamLite weights are licensed CC BY-NC 4.0 with additional upstream use restrictions. Review the model card and `WEIGHTS_LICENSE`. Use only for permitted non-commercial/research purposes. Keep exported model files private and do not commit them to this public repository.

Choose **Runtime → Change runtime type → T4 GPU** if available. The notebook checks for CUDA before starting.

In [ ]:
import torch, sys
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime in Colab, then rerun this cell.')
!nvidia-smi

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/ByteVisionLab/DreamLite.git
!git clone --depth 1 --branch feature/dreamlite-android-integration https://github.com/tensor004/Repix.git
!mkdir -p /content/DreamLite/tools/dreamlite_onnx
!cp -r /content/Repix/tools/dreamlite_onnx/. /content/DreamLite/tools/dreamlite_onnx/

In [ ]:
# Keep Colab's preinstalled CUDA-enabled PyTorch. Install the remaining pinned runtime/export dependencies.
%pip install -q 'diffusers==0.37.0' 'transformers==4.57.3' accelerate safetensors einops peft onnx onnxruntime huggingface_hub

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id='carlofkl/DreamLite-mobile',
    revision='main',
    allow_patterns=['unet/*', 'vae/*'],
    local_dir='/content/DreamLite/models/DreamLite-mobile',
)
print('Downloaded only the UNet and VAE component folders.')

In [ ]:
%cd /content/DreamLite
!python tools/dreamlite_onnx/export_mobile.py --model-id models/DreamLite-mobile --output-dir artifacts/dreamlite-onnx --device cuda --validate

In [ ]:
from pathlib import Path
import json
report = json.loads(Path('artifacts/dreamlite-onnx/export_report.json').read_text())
print(json.dumps(report, indent=2))
assert all(x.get('passed') for x in report['validation'].values()), 'At least one ONNX parity check failed.'
print('All exported component graphs passed desktop ONNX Runtime CPU parity.')

In [ ]:
!cd /content/DreamLite && zip -qr /content/dreamlite-onnx.zip artifacts/dreamlite-onnx
from google.colab import files
files.download('/content/dreamlite-onnx.zip')

## What to send back

If the notebook completes, keep the ZIP private and share the `export_report.json` contents here (or upload the report file). Do not upload model weights. If it fails, share the full error from the failed cell. The next engineering gate is the Qwen3-VL image-conditioned hidden-state path and then Qualcomm QNN/device validation.